# *<center>V03 · Pseudopotential vs direct RF</center>*

**Purpose.** `pe_surface` draws the Dehmelt effective potential — a
*visualization concept*; the tracer always integrates the real
time-varying field. This notebook proves the picture is quantitatively
trustworthy where its own docstring claims validity (adiabatic regime,
Mathieu q ≲ 0.4), measures exactly how it degrades beyond, and anchors
the direct-RF integrator itself against **exact Floquet theory**
(Mathieu / Meissner monodromy) at the per-mille level.

```
PROVENANCE
  origin   : validation series
  template : V01/V05/V06 (assumptions / methods / citations; bands
             calibrated from measurement, never assumed)
```

### Conventions
* **Units are mm, V, µs, eV**; frequencies in MHz (cycles/µs).
* **CAPITALS are parameters you may change**; lower-case is computed.
* Every threshold is declared before its measurement and asserted.

---

### Assumptions (explicit)
1. **q is measured, not assumed.** The Mathieu parameter comes from the
   *solved* field's pe curvature (q = 2√2·ω_pe/Ω, exact inversion of the
   Dehmelt relation, linear in V) — never from ideal-rod textbook
   formulas. Round rods at r_rod = 1.148·r0 minimize but do not
   eliminate higher multipoles [4]; using the measured curvature makes
   the small-amplitude comparison exact by construction.
2. **Vacuum, single ion.** Collisions off; every flight is one cold ion
   (thermal statistics are V05's subject, not this one).
3. **Frequency by zero-crossing count** of the RF-period-smoothed
   coordinate. FFT-peak interpolation carries up to ~2% bin-placement
   bias at these record lengths (measured in the prototype session: a
   56.25 kHz line landing on exactly 9 window cycles read 0.06% while an
   off-bin line read 2% — the estimator, not the physics); zero-crossing
   counting with interpolated end-crossings is bin-free.
4. **The exact anchor is the monodromy matrix** of the Hill equation
   x'' + 2q·m(2τ)·x = 0 over one period (RK4, 4000 steps), with
   m = cos for sinusoidal drive (Mathieu) and m = sign(cos) for square
   drive (Meissner): cos(πβ) = tr(M)/2, f_sec = β·f_RF/2. Stable iff
   |tr(M)| ≤ 2 (boundary q = 0.908 for Mathieu a = 0) [2,3].


## The instrument, before any statistics

The device this notebook flies, drawn from the **solver's own electrode mask** (not a redrawing) with example ion paths exactly as flown. You are looking at the four rods with the RF field at one instant, and example ions showing the two-timescale motion: slow secular oscillation with fast micromotion riding on it.

Deck: `examples/quadrupole_stl_rods_transport.json`. A geometry figure is not decoration — if the picture and the solved model can disagree, every number below is unverifiable.

In [ ]:
from ion_gym.viz.nb_panels import show_instrument_panel
# banked panel when present, else rendered live from the deck
# of record (self-contained on a public checkout).
show_instrument_panel('examples/quadrupole_stl_rods_transport.json', banked='panel_quadrupole.png', height=520)


In [ ]:
import json
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '..')

from ion_gym.io.sim_spec import (SimSpec, GeometrySpec, ElectrodeSpec,
                                 ShapeSpec, SourceSpec, IntegrationSpec,
                                 CollisionSpec, RFGroupSpec)
from ion_gym.physics.symmetry import SymmetrySpec
from ion_gym.physics.sim_build import build_run

AMU = 1.66053906660e-27
E_CHG = 1.602176634e-19

# ---- parameters (CAPITALS are yours to change) ----
PITCH_MM = 0.1                 # raster pitch [mm]
R0_MM = 3.0                    # field radius
ROD_FACTOR = 1.148             # r_rod/r0, the dodecapole-minimizing ratio
F_RF_MHZ = 1.0
MZ = 100.0
T_FLY_US = 160.0
DT_NS = 2.0
X0_MM = 0.35    # launch offset [mm]. Was 0.5: at the top rung
                # (q=0.844) that orbit swings to within h/2 of the rod
                # surface, and the certified pass rode a raster phase —
                # the A7 domain snap's 0.012 mm frame shift flipped the
                # graze into a strike (measured). Mathieu is
                # LINEAR: frequency is amplitude-independent, so a
                # smaller launch changes no gate physics, only restores
                # clearance (~0.9 mm at the top rung).                    # launch offset for the frequency ladder
V_LADDER = [15.0, 30.0, 50.0, 80.0]        # q ~ 0.16 .. 0.84
SEED_NOTE = "deterministic (single cold ion, no RNG)"

RROD = ROD_FACTOR * R0_MM
OM = 2 * np.pi * F_RF_MHZ      # rad/us
M_KG = MZ * AMU

def quad_spec(v0, waveform="sin", t_max=T_FLY_US, x0=X0_MM, ke=0.0,
              pe_mode=None, dt_ns=DT_NS):
    rf = [RFGroupSpec(name="RFX", waveform=waveform,
                      frequency_hz=F_RF_MHZ * 1e6, amplitude_v=v0,
                      phase_deg=0.0),
          RFGroupSpec(name="RFY", waveform=waveform,
                      frequency_hz=F_RF_MHZ * 1e6, amplitude_v=v0,
                      phase_deg=180.0)]
    if pe_mode:
        for g in rf:
            g.pe_mode = pe_mode
    cdist = R0_MM + RROD
    W = 2 * (R0_MM + 2 * RROD) + 2.0
    # A7: the DOMAIN is a lattice quantity — an integer number of cells,
    # exactly; the physical envelope's remainder is absorbed at the
    # OUTER WALLS (snap UP), never silently and never at a symmetry
    # plane. Metal stays in mm and rasterizes (the measured-hardware
    # carve-out). The raw envelope is 21.776 mm, which the loader
    # refuses at 0.1 mm/gu; snapped it is 21.8 mm = 218 cells exactly.
    W = float(np.ceil(W / PITCH_MM - 1e-9) * PITCH_MM)
    ctr = W / 2
    def rod(name, dx, dy, grp):
        return ElectrodeSpec(name=name, dc=0.0, rf_groups=[grp], shapes=[
            ShapeSpec(type="ellipse", params={
                "cx_mm": ctr + dx, "cy_mm": ctr + dy,
                "rx_mm": RROD, "ry_mm": RROD})])
    geom = GeometrySpec(
        width_mm=W, height_mm=W, mm_per_gu=PITCH_MM,
        symmetry=SymmetrySpec(coords="xyz"), rf_groups=rf,
        electrodes=[rod("xp", cdist, 0, "RFX"), rod("xm", -cdist, 0, "RFX"),
                    rod("yp", 0, cdist, "RFY"), rod("ym", 0, -cdist, "RFY")])
    return SimSpec(
        name="V03 quad", geometry=geom,
        source=SourceSpec(n_ions=1, distribution="point", x0_mm=ctr + x0,
                          y0_mm=ctr, ke_lo=ke, ke_hi=ke,
                          direction=[1.0, 0.0, 0.0], temperature_k=0.0,
                          mz_list=[MZ], tob_span_us=0.0),
        collisions=CollisionSpec(enabled=False),
        integration=IntegrationSpec(t_max_us=t_max, dt_ns=dt_ns,
                                    rec_every=25))

def fly_one(sp):
    errs = sp.validate()
    assert not errs, errs
    # FIELD GUARD (V06 lesson): the drive must be ON the spec
    amps = [g.amplitude_v for g in sp.geometry.rf_groups]
    assert all(a != 0 for a in amps), "RF amplitude missing from spec!"
    model, f, cols, births = build_run(sp)
    tr, status = f(0)
    return model, np.asarray(tr), {cc: j for j, cc in enumerate(cols)}, status

def f_zero_cross(tr, ci, ref, f_rf_mhz=F_RF_MHZ, col="x"):
    """Secular frequency (MHz) by zero-crossing count of the
    RF-period-smoothed coordinate; interpolated end crossings."""
    t = tr[:, ci["t"]]
    x = tr[:, ci[col]] - ref
    dt = t[1] - t[0]
    n = max(1, int(round(1.0 / f_rf_mhz / dt)))
    xs = np.convolve(x - x.mean(), np.ones(n) / n, mode="valid")
    ts = t[n - 1:]
    s = np.signbit(xs)
    z = np.nonzero(s[1:] != s[:-1])[0]
    if len(z) < 4:
        return np.nan
    tz = []
    for i in (z[0], z[-1]):
        a, b = xs[i], xs[i + 1]
        tz.append(ts[i] + dt * a / (a - b))
    return (len(z) - 1) / (2 * (tz[1] - tz[0]))

def hill_beta(q, square=False):
    """Floquet exponent beta of x'' + 2 q m(2 tau) x = 0 over one
    period (RK4): cos(pi beta) = tr(M)/2. m = cos (Mathieu) or
    sign(cos) (Meissner)."""
    tau = np.linspace(0, np.pi, 4001)
    dtau = tau[1] - tau[0]
    def rhs(ti, y):
        mod = np.sign(np.cos(2 * ti)) if square else np.cos(2 * ti)
        return np.array([y[1], 2 * q * mod * y[0]])
    def step(y):
        for ti in tau[:-1]:
            k1 = rhs(ti, y)
            k2 = rhs(ti + dtau / 2, y + dtau / 2 * k1)
            k3 = rhs(ti + dtau / 2, y + dtau / 2 * k2)
            k4 = rhs(ti + dtau, y + dtau * k3)
            y = y + dtau / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        return y
    y1 = step(np.array([1.0, 0.0]))
    y2 = step(np.array([0.0, 1.0]))
    return np.arccos(np.clip((y1[0] + y2[1]) / 2, -1, 1)) / np.pi

def pe_cut(model, mz, at, axis="x"):
    xs, ys, PE, ele = model.pe_surface(mz=mz)
    if axis == "x":
        cy = np.argmin(np.abs(ys - at))
        return xs, PE[:, cy]
    cx = np.argmin(np.abs(xs - at))
    return ys, PE[cx, :]

def pe_vertex_fit(u, pe, u_guess, fit_mm):
    """Parabola fit around a well: (f_pe_MHz for M_KG, vertex, d2).
    Interpolation-grade — immune to the grid-offset quantization that
    biased the naive node readout by up to 11% in prototyping."""
    m0 = np.abs(u - u_guess) <= fit_mm
    a, b, _cc = np.polyfit(u[m0], pe[m0], 2)
    return -b / (2 * a), 2 * a

print("helpers ready;", SEED_NOTE)

In [ ]:
# ---- solve-cache control ------------------------------------------
# Banked field bases make re-runs fast but can serve STALE fields
# after a geometry edit. Set True to clear the cache and force fresh
# solves for this run.
CLEAR_FIELD_CACHE = False
if CLEAR_FIELD_CACHE:
    import shutil
    from pathlib import Path
    from ion_gym.io.fa_cache import DEFAULT_ROOT
    _cache = Path(DEFAULT_ROOT)
    if 'ion_gym' not in _cache.name:
        raise ValueError(f'refusing to clear {_cache} — not an ion_gym '
                         'cache dir (check ION_GYM_CACHE)')
    if _cache.exists():
        shutil.rmtree(_cache)
        print(f'cleared solve cache: {_cache}')
    else:
        print(f'solve cache already empty: {_cache}')


## 1 · The instrument and its pseudopotential

Four round rods (r_rod = 1.148·r0) driven ±V·sin(Ωt) as pairs, solved on
the ordinary planar route. The pe surface below is what `pe_surface`
draws; both principal cuts are shown (x and y are equivalent by
symmetry — shown to prove it, per the multi-axis rule).

In [ ]:
t0 = time.time()
sp0 = quad_spec(V_LADDER[1])
CTR = sp0.geometry.width_mm / 2
model0, tr0, ci0, st0 = fly_one(sp0)
xs, ys, PE, ele = model0.pe_surface(mz=MZ)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
im = axes[0].pcolormesh(xs, ys, PE.T, shading="auto", cmap="viridis",
                        vmax=np.nanpercentile(PE[~ele.astype(bool)], 99))
axes[0].contour(xs, ys, ele.T, levels=[0.5], colors="w", linewidths=0.7)
axes[0].set_title("pe surface (eV), electrodes white", fontsize=9)
axes[0].set_xlabel("x (mm)")
axes[0].set_ylabel("y (mm)")
plt.colorbar(im, ax=axes[0])
for ax, axis in ((axes[1], "x"), (axes[2], "y")):
    u, pe1 = pe_cut(model0, MZ, CTR, axis)
    m = np.abs(u - CTR) < R0_MM
    ax.plot(u[m] - CTR, pe1[m], lw=1.2)
    ax.set_xlabel(f"{axis} - center (mm)")
    ax.set_ylabel("PE (eV)")
    ax.set_title(f"{axis} cut through center", fontsize=9)
plt.tight_layout()
plt.show()
print(f"[{time.time()-t0:.0f}s] geometry + surface")

## 2 · The frequency ladder: measured vs Mathieu vs Dehmelt

One cold ion launched 0.5 mm off-axis at each drive amplitude. q comes
from the measured curvature (assumption 1); the **exact** prediction is
β(q)·f_RF/2 from the monodromy; the **Dehmelt** prediction is the pe
curvature frequency itself.

**Declared bands** (calibrated from the prototype measurements, ~3×
headroom on the observed residuals):
* **A (integrator vs exact theory):** |f_meas/f_Mathieu − 1| < **0.02**
  at every stable ladder point (measured ≤ 0.006).
* **B (Dehmelt validity):** |f_meas/f_pe − 1| < **0.03** for q ≤ 0.35
  (measured ≤ 0.021), and the departure grows **monotonically** with q,
  exceeding +5% by q ≈ 0.53 and +20% by q ≈ 0.84 — the docstring's
  "adiabatic regime, q ≲ 0.4" made quantitative.

In [ ]:
rows = []
t0 = time.time()
for V0 in V_LADDER:
    sp = quad_spec(V0)
    m, tr, ci, st = fly_one(sp)
    f_m = f_zero_cross(tr, ci, CTR)
    u, pe1 = pe_cut(m, MZ, CTR, "x")
    v0x, d2 = pe_vertex_fit(u, pe1, CTR, 0.6)
    f_pe = np.sqrt(max(d2, 0) * E_CHG * 1e6 / M_KG) * 1e-6 / (2 * np.pi)
    q = (f_pe * 2 * np.pi) * 2 * np.sqrt(2) / OM
    f_mat = hill_beta(q) * OM / 2 / (2 * np.pi)
    rows.append((V0, q, f_m, f_pe, f_mat))
    print(f"  V={V0:5.0f}  q={q:.4f}  f_meas={f_m*1e3:7.2f} kHz  "
          f"meas/Mathieu={f_m/f_mat:.4f}  meas/Dehmelt={f_m/f_pe:.4f}")
print(f"[ladder {time.time()-t0:.0f}s]")

qs = np.array([r[1] for r in rows])
r_mat = np.array([r[2] / r[4] for r in rows])
r_pe = np.array([r[2] / r[3] for r in rows])
PASS_A = bool(np.all(np.abs(r_mat - 1) < 0.02))
low = qs <= 0.35
dep = r_pe - 1
PASS_B = (bool(np.all(np.abs(dep[low]) < 0.03))
          and bool(np.all(np.diff(dep) > 0))
          and dep[qs > 0.5][0] > 0.05 and dep[-1] > 0.20)
print("PASS" if PASS_A else "FAIL",
      "— A: direct integration matches exact Floquet theory to <2% "
      "across the full stable range")
print("PASS" if PASS_B else "FAIL",
      "— B: Dehmelt <3% inside its declared q<~0.4 domain; departure "
      "monotone, +5%@q~0.53, +20%@q~0.84")
assert PASS_A and PASS_B

fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot(qs, r_mat, "o-", label="meas / Mathieu (exact)")
ax.plot(qs, r_pe, "s-", label="meas / Dehmelt (pe_surface)")
ax.axhline(1.0, color="k", lw=0.6)
ax.axvspan(0, 0.4, alpha=0.12, color="g",
           label="docstring validity q<~0.4")
ax.set_xlabel("Mathieu q (measured curvature)")
ax.set_ylabel("frequency ratio")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3 · The stability boundary

Mathieu (a = 0) is stable for q < 0.908. Launched at 0.1 mm (the
near-boundary amplitude swell would reach the rods from a 0.5 mm launch
even on the stable side — measured in prototyping):

* **C:** q = 0.86 survives ≥ 90% of the flight; q = 0.95 is lost within
  10% of it.

In [ ]:
V_PER_Q = None
# q is linear in V (curvature exact inversion): calibrate from the ladder
V_PER_Q = V_LADDER[-1] / qs[-1]
res = {}
for q_t in (0.86, 0.95):
    sp = quad_spec(q_t * V_PER_Q, t_max=120.0, x0=0.1)
    m, tr, ci, st = fly_one(sp)
    res[q_t] = tr[-1, ci["t"]]
    print(f"  q={q_t}: flew {res[q_t]:.1f} / 120 us")
PASS_C = res[0.86] > 0.9 * 120.0 and res[0.95] < 0.1 * 120.0
print("PASS" if PASS_C else "FAIL",
      "— C: survival brackets the q=0.908 Mathieu boundary")
assert PASS_C

## 4 · Square (digital) drive: the π²/6 harmonic sum, and Meissner

`pe_surface` claims square drives under `pe_mode='pseudo'` carry the
harmonic-sum factor π²/6 = Σ_odd (4/nπ)²/n². Validated two ways at an
adiabatic operating point (plateau q ≈ 0.13):

* **D1 (surface):** curvature ratio square/sin at equal amplitude equals
  **π²/6** to < 10⁻³ (implementation-exact in prototyping).
* **D2 (dynamics):** |f_meas/f_Meissner − 1| < **0.01** against the exact
  square-wave monodromy (measured 0.0005), and |f_meas/f_pe − 1| < 0.03.

In [ ]:
V_SQ = 12.0
sp_sq = quad_spec(V_SQ, waveform="square", pe_mode="pseudo")
m_sq, tr_q, ci_q, _ = fly_one(sp_sq)
f_m_sq = f_zero_cross(tr_q, ci_q, CTR)
u, pe_sq1 = pe_cut(m_sq, MZ, CTR, "x")
_v, d2_sq = pe_vertex_fit(u, pe_sq1, CTR, 0.6)
sp_sin = quad_spec(V_SQ)
m_sn, _t, _c, _s = fly_one(sp_sin)
u, pe_sn1 = pe_cut(m_sn, MZ, CTR, "x")
_v, d2_sn = pe_vertex_fit(u, pe_sn1, CTR, 0.6)
factor = d2_sq / d2_sn
f_pe_sq = np.sqrt(d2_sq * E_CHG * 1e6 / M_KG) * 1e-6 / (2 * np.pi)
q_plateau = V_SQ / V_PER_Q
f_meis = hill_beta(q_plateau, square=True) * OM / 2 / (2 * np.pi)
print(f"  surface factor square/sin = {factor:.5f} "
      f"(pi^2/6 = {np.pi**2/6:.5f})")
print(f"  f_meas = {f_m_sq*1e3:.2f} kHz; meas/Meissner = "
      f"{f_m_sq/f_meis:.4f}; meas/Dehmelt(pi^2/6) = {f_m_sq/f_pe_sq:.4f}")
PASS_D = (abs(factor - np.pi**2 / 6) < 1e-3
          and abs(f_m_sq / f_meis - 1) < 0.01
          and abs(f_m_sq / f_pe_sq - 1) < 0.03)
print("PASS" if PASS_D else "FAIL",
      "— D: harmonic-sum surface exact; dynamics match the exact "
      "Meissner monodromy to <1%")
assert PASS_D

## 5 · Well depth: turning points and the micromotion share

An ion born **at the well center** with kinetic energy KE must turn where
the pseudopotential contour reaches KE — energy conservation in the
effective potential. The raw trajectory overshoots by the micromotion
riding the secular envelope, whose fractional share is ≈ q/2·(1 + q/2)
to next order.

* **E:** |secular turning point / pe contour − 1| < **0.03** (measured
  0.006 with interpolated contour + vertex fit — the naive grid-node
  readout carries up to 11% quantization, the prototype's lesson);
  micromotion share within **[0.7, 1.3]·q/2·(1+q/2)** (measured 0.180
  vs 0.183).

In [ ]:
KE_EV = 0.02
V_TP = 30.0
spt = quad_spec(V_TP, x0=0.0, ke=KE_EV)
m, tr, ci, st = fly_one(spt)
t = tr[:, ci["t"]]
x = tr[:, ci["x"]] - CTR
dt = t[1] - t[0]
n = max(1, int(round(1.0 / F_RF_MHZ / dt)))
x_sec = np.convolve(x, np.ones(n) / n, mode="valid")
u, pe1 = pe_cut(m, MZ, CTR, "x")
v0x, d2 = pe_vertex_fit(u, pe1, CTR, 0.6)
f_pe = np.sqrt(d2 * E_CHG * 1e6 / M_KG) * 1e-6 / (2 * np.pi)
q_tp = (f_pe * 2 * np.pi) * 2 * np.sqrt(2) / OM
pe_rel = pe1 - np.interp(v0x, u, pe1)
ui = np.linspace(v0x, v0x + 1.5, 3001)
x_pred = ui[np.argmax(np.interp(ui, u, pe_rel) >= KE_EV)] - v0x
x_meas = np.percentile(np.abs(x_sec), 99.9)
x_raw = np.percentile(np.abs(x), 99.9)
share = x_raw / x_meas - 1
share_pred = q_tp / 2 * (1 + q_tp / 2)
print(f"  secular turn {x_meas:.4f} mm vs contour {x_pred:.4f} "
      f"(ratio {x_meas/x_pred:.4f})")
print(f"  micromotion share {share:.3f} vs q/2(1+q/2) = {share_pred:.3f}")
PASS_E = (abs(x_meas / x_pred - 1) < 0.03
          and 0.7 * share_pred < share < 1.3 * share_pred)
print("PASS" if PASS_E else "FAIL",
      "— E: energy conservation in the effective potential; micromotion "
      "share at its adiabatic value")
assert PASS_E

# ---- deck-inherited physics: visible and overridable -----------------
# The deck loaded below supplies the drive, the ion, the gas and the
# integration settings. Leave an entry None to INHERIT it from the deck;
# set one to OVERRIDE. Whatever ends up in force is printed, so this
# notebook's output always states its own operating point.
# (One namespaced dict, not loose globals: the first version used bare
# names like KE_EV and N_IONS, which collided with the parameters these
# notebooks already own -- and silently changed them.)
DECK_OVERRIDES = dict(
    rf_v=None, rf_f=None,        # confining-drive amplitude (V) / freq (Hz)
    mz_list=None, charge=None,   # e.g. [622.0] / 1
    ke_ev=None,                  # (lo, hi) eV
    source_t_k=None,             # K, thermal spread of initial velocities
    n_ions=None,                 # ions flown
    gas_on=None, gas=None,       # True/False / e.g. "N2"
    p_torr=None, gas_t_k=None,   # buffer-gas pressure (Torr) / temp (K)
    dt_ns=None, t_max_us=None,   # integration step (ns) / flight time (us)
)


## 6 · A real instrument well: the SLIM confinement slice

The shipped `slim_tetramer_confinement_2-d_acrossxgap` example (RF wire
pairs at 0.8 MHz) — the well the tetramer actually confines in. Two
instrument-relevant facts, both measured:

1. **Wire wells are anharmonic.** The fitted curvature depends on the
   fit window (hardening quartic content). The honest harmonic
   prediction is the *vertex* curvature, obtained by extrapolating the
   window ladder to zero width; the measured small-amplitude frequency
   must match it, and frequency must **rise** with amplitude (hardening
   sign).
2. **Pseudopotential validity is the low-mass cutoff.** At the
   example's own m/z 300 this drive puts the well at q_eff ≈ 0.9 —
   outside the adiabatic picture — and the cold ion is **lost within
   microseconds**. At m/z 1200 (q_eff ≈ 0.13) it is stably trapped.

* **F1:** |f_meas(small amp) / f_pe(vertex, window→0) − 1| < **0.05**
  (the linear window→0 extrapolation is itself window-set-sensitive
  at the grid pitch — measured 0.030 here, 0.003 with a different
  window set in prototyping; the 5% band carries that spread).
* **F2:** f(dy=0.08) > f(dy=0.02) (hardening sign).
* **F3:** m/z 300 is lost within a **tenth of the survival horizon** —
  t(lost)/t(survived) < 0.1 (ratio form; the old absolute
  16 µs clock was birth-phase-sensitive at q_eff ≈ 0.9); m/z 1200
  survives the full flight.
  (No scalar q_eff is asserted: a hardening wire well has no single q —
  the harmonic-scaled value printed is a lower-bound annotation only.)

In [ ]:
# Repo-relative root: every path in this cell derives from it, so the
# notebook runs on any machine and in any cell order.
from pathlib import Path as _P
from ion_gym.io.paths import repo_root as _rr
ROOT = _P(_rr())
d = json.load(open(str(ROOT / 'examples/slim_tetramer_confinement_2-d_acrossxgap.json')))
base = SimSpec.from_dict(d)
X0S, Y0S = base.source.x0_mm, base.source.y0_mm
F_RF_SLIM = base.geometry.rf_groups[0].frequency_hz / 1e6
MZ_HI, MZ_LO = 1200.0, base.source.mz_list[0]

def slim_fly(mz, dy, t_max=240.0):
    sp = SimSpec.from_dict(d)
    # SEEDED: Gate F's thresholds are on single-ion loss
    # times, which under the random-by-default rule rode the birth
    # draw — a certified gate must not be a seed lottery.
    sp.source.seed = 0
    sp.collisions.enabled = False
    sp.source.distribution = "point"
    sp.source.n_ions = 1
    sp.source.mz_list = [mz]
    sp.source.y0_mm = Y0S + dy
    sp.source.ke_lo = sp.source.ke_hi = 0.0
    sp.source.temperature_k = 0.0
    sp.source.tob_span_us = 0.0
    sp.integration = IntegrationSpec(t_max_us=t_max, dt_ns=DT_NS,
                                     rec_every=25)
    return fly_one(sp)

# vertex curvature by window-ladder extrapolation (anharmonic well)
m_s, _tr, _ci, _st = slim_fly(MZ_HI, 0.02, t_max=10.0)
u, pe_y = pe_cut(m_s, MZ_HI, X0S, "y")
wins = np.array([0.08, 0.15, 0.30])
d2s = []
for wv in wins:
    _v, d2w = pe_vertex_fit(u, pe_y, Y0S, wv)
    d2s.append(d2w)
d2s = np.array(d2s)
sl, d2_0 = np.polyfit(wins, d2s, 1)      # linear window -> 0 intercept
m_hi = MZ_HI * AMU
f_pe0 = np.sqrt(max(d2_0, 0) * E_CHG * 1e6 / m_hi) * 1e-6 / (2 * np.pi)
wdict = {float(w): round(float(v), 4) for w, v in zip(wins, d2s)}
print(f"  curvature vs window: {wdict} "
      f"-> vertex (w->0) {d2_0:.4f} eV/mm2, f_pe = {f_pe0*1e3:.2f} kHz")

fs = {}
for dy in (0.02, 0.08):
    m2, tr2, ci2, st2 = slim_fly(MZ_HI, dy)
    fs[dy] = f_zero_cross(tr2, ci2, Y0S, f_rf_mhz=F_RF_SLIM, col="y")
    print(f"  m/z {MZ_HI:.0f} dy={dy}: t_end={tr2[-1, ci2['t']]:.0f} us, "
          f"f = {fs[dy]*1e3:.2f} kHz  (meas/vertex-pe = "
          f"{fs[dy]/f_pe0:.4f})")
m3, tr3, ci3, st3 = slim_fly(MZ_LO, 0.02, t_max=160.0)
t_end_lo = tr3[-1, ci3["t"]]
# NOT asserted: a wire well has no single q (it hardens off-vertex).
# The harmonic-scaled value below is a LOWER-BOUND estimate only; the
# local q at the excursion amplitudes exceeds it. The asserted facts
# are the measured loss and survival times themselves.
q_eff_est = 2 * np.sqrt(2) * (fs[0.02] * MZ_HI / MZ_LO) / F_RF_SLIM
print(f"  m/z {MZ_LO:.0f}: t_end = {t_end_lo:.1f} us (harmonic-scaled "
      f"q_eff >~ {q_eff_est:.2f}) — the low-mass cutoff, measured")
# Gate F is FOUR separate claims. A bare `assert PASS_F` reports which of
# them failed: none. Check them individually, print each with its measured
# value, margin and threshold, and raise naming only the ones that broke
# (this assert once fired with no diagnostic).
_checks = [
    ("pe predicts wire-well frequency",
     abs(fs[0.02] / f_pe0 - 1) < 0.05,
     f"|f_meas/f_pe - 1| = {abs(fs[0.02]/f_pe0 - 1):.4f}", "< 0.05"),
    ("hardening sign (larger excursion -> higher f)",
     fs[0.08] > fs[0.02],
     f"f(dy=0.08) = {fs[0.08]*1e3:.2f} kHz vs f(dy=0.02) = "
     f"{fs[0.02]*1e3:.2f} kHz", "f(0.08) > f(0.02)"),
    # RATIO FORM: "promptly" means within a tenth
    # of the demonstrated survival horizon, t_lo/t_hi < 0.1. Replaces the
    # absolute < 16 us clock, whose measured provenance was unrecoverable
    # from the record and whose value is birth-phase-sensitive at
    # q_eff ~ 0.9 (a chaotic ejection); the ratio is scale-honest and
    # carries its own operating point in both numbers.
    ("low-mass ion is lost promptly (ratio)",
     t_end_lo / tr2[-1, ci2["t"]] < 0.1,
     f"t_end(m/z {MZ_LO:.0f}) / t_end(m/z {MZ_HI:.0f}) = {t_end_lo:.1f} / "
     f"{tr2[-1, ci2['t']]:.1f} us = {t_end_lo / tr2[-1, ci2['t']]:.3f}",
     "< 0.1"),
    ("high-mass ion survives",
     tr2[-1, ci2["t"]] > 230.0,
     f"t_end(m/z {MZ_HI:.0f}) = {tr2[-1, ci2['t']]:.1f} us", "> 230 us"),
]
for _name, _ok, _meas, _thr in _checks:
    print(f"  [{'PASS' if _ok else 'FAIL'}] {_name}: {_meas}  (needs {_thr})")
PASS_F = all(_ok for _, _ok, _, _ in _checks)
print("PASS" if PASS_F else "FAIL",
      "— F: vertex-extrapolated pe predicts the wire-well frequency; "
      "hardening sign correct; pseudopotential validity IS the "
      "low-mass cutoff")
if not PASS_F:
    _bad = "; ".join(f"{_n} ({_m}, needs {_t})"
                     for _n, _o, _m, _t in _checks if not _o)
    raise AssertionError(
        f"Gate F failed on: {_bad}. These are MEASURED values from this "
        f"run -- compare them against the thresholds above before changing "
        f"anything; a threshold that no longer holds may mean the deck, the "
        f"dt, or the physics changed, and which one is named here.")
# Report what came from the deck and apply any override set above.
from ion_gym.io.deck_params import apply_deck_overrides
apply_deck_overrides(base, **DECK_OVERRIDES)


### Gate F, seen rather than tallied

The four checks above are printed verdicts. Both of the facts they assert
are *shapes*, and a shape is worth drawing:

**Left:** the wire well itself, the solved pseudopotential along y through
the confinement axis, with the parabola implied by the vertex curvature
laid over it. Where the two separate is the anharmonicity — the reason the
fitted curvature depends on the fit window at all, and the reason the
harmonic prediction must be read at the vertex rather than from any one
window.

**Right:** the low-mass cutoff as it actually happens. The heavy ion
oscillates in the well for the full flight; the light one, at the same
drive, leaves. That is what "pseudopotential validity is the low-mass
cutoff" means, and it is far more legible as two traces than as two
end-times in a print statement.

In [ ]:
# RUN CELL -- Gate F drawn, through the framework (viz_core.lineout_figure).
# Uses the SAME arrays the printed verdicts came from -- nothing is
# reflown for the picture, so the figure cannot disagree with the gate.
from ion_gym.viz.viz_core import lineout_figure
from IPython.display import Image as _PNG, display as _display
import io as _io, matplotlib.pyplot as _plt

def _show(fig, dpi=110):
    _b = _io.BytesIO()
    fig.savefig(_b, format="png", dpi=dpi, bbox_inches="tight")
    _display(_PNG(_b.getvalue()))
    _plt.close(fig)

# ---- the well, and the parabola its vertex curvature implies ----------
# Drawn over +-0.5 mm about the well centre: wide enough that the quartic
# content is visible, narrow enough that the neighbouring wire wells do
# not dominate the axis.
_win_mm = 0.5
_m = np.abs(u - Y0S) <= _win_mm
_pe0 = float(np.interp(Y0S, u, pe_y))
_para = _pe0 + 0.5 * d2_0 * (u[_m] - Y0S) ** 2          # vertex curvature
_para_w = _pe0 + 0.5 * d2s[-1] * (u[_m] - Y0S) ** 2     # widest-window fit
fig_well = lineout_figure(
    [(u[_m] - Y0S, pe_y[_m], "solved pseudopotential", "line"),
     (u[_m] - Y0S, _para, f"vertex parabola (d2 -> 0 window = {d2_0:.3f} eV/mm2)",
      "dash"),
     (u[_m] - Y0S, _para_w,
      f"parabola from the {wins[-1]:.2f} mm window ({d2s[-1]:.3f} eV/mm2)",
      "dots")],
    xlabel="y - well centre (mm)", ylabel="pseudopotential (eV)",
    title="The SLIM wire well is anharmonic: curvature depends on the window",
    operating_point=(f"slim_tetramer_confinement_2-d_acrossxgap | m/z "
                     f"{MZ_HI:.0f} | RF {F_RF_SLIM:.2f} MHz | vertex f_pe "
                     f"= {f_pe0*1e3:.2f} kHz"))
_show(fig_well)

# ---- the low-mass cutoff, as trajectories -----------------------------
fig_cut = lineout_figure(
    [(tr2[:, ci2["t"]], tr2[:, ci2["y"]] - Y0S,
      f"m/z {MZ_HI:.0f}: confined, t_end {tr2[-1, ci2['t']]:.0f} us", "line"),
     (tr3[:, ci3["t"]], tr3[:, ci3["y"]] - Y0S,
      f"m/z {MZ_LO:.0f}: lost, t_end {t_end_lo:.1f} us", "line")],
    xlabel="time (us)", ylabel="y - well centre (mm)",
    title="Pseudopotential validity IS the low-mass cutoff",
    operating_point=(f"same deck, same drive ({F_RF_SLIM:.2f} MHz), same "
                     f"birth offset dy = 0.08 / 0.02 mm | collisions off | "
                     f"seed 0 | ratio t_lo/t_hi = "
                     f"{t_end_lo / tr2[-1, ci2['t']]:.3f}"))
_show(fig_cut)


---
## Summary

| Gate | Claim | Band | Result |
|---|---|---|---|
| A | direct RF vs exact Mathieu | <2% | ~0.1–0.6% |
| B | Dehmelt inside q≲0.4 | <3%, departure monotone | +0.5→+27% over q 0.16→0.84 |
| C | stability boundary | brackets q=0.908 | 0.86 lives / 0.95 lost |
| D | square π²/6 + Meissner | 10⁻³ / <1% | exact / 0.05% |
| E | turning point + micromotion | <3% / q-scaled | 0.6% / 0.180 vs 0.183 |
| F | SLIM wire well | <5% vertex; hardening; cutoff | ✓ all |

**What this buys the instrument work:** the pe surface can be trusted as
a *design* tool wherever local q ≲ 0.4, with a measured error budget
beyond; wire-well curvature must be read at the vertex (quartic
hardening is large at wire scale); and the SLIM low-mass cutoff is not a
tuning accident but the adiabaticity boundary itself.

### Citations
[1] H. G. Dehmelt, *Adv. At. Mol. Phys.* **3**, 53 (1967) — the
effective-potential picture.
[2] N. W. McLachlan, *Theory and Application of Mathieu Functions*
(1947) — β(q), stability chart.
[3] W. Paul, *Rev. Mod. Phys.* **62**, 531 (1990) — quadrupole traps and
the a–q diagram.
[4] D. R. Denison, *J. Vac. Sci. Technol.* **8**, 266 (1971) — round-rod
r_rod/r0 = 1.148 (recent optimizations differ slightly; 1.148 is the
conventional value and its residual multipoles are irrelevant here by
assumption 1).
[5] D. Gerlich, *Adv. Chem. Phys.* **82**, 1 (1992) — adiabaticity
parameter and validity of the effective potential.
[6] E. Meissner (1918) — square-wave Hill equation; monodromy per [2].
[7] Ibrahim, Smith *et al.* — SLIM traveling-wave structures
(*Analyst* **142**, 1010 (2017)) — the wire-pair confinement well of §6.


## Read-out

The pseudopotential is an approximation with a stated domain of validity (adiabaticity: the field must change little over one RF cycle as the ion moves). This comparison maps where it holds and where it fails — and the failure region is exactly where a design must be checked with the full RF integration rather than the effective potential.